In [1]:
"""
quality_checks
Runs automated data quality checks across all clean_* tables.
Outputs results to ../data/audit_log.csv
"""

import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import os

DB_PATH  = "cdr_monitor.db"
LOG_PATH = "../data/audit_log.csv"

os.makedirs("data", exist_ok=True)

conn    = sqlite3.connect(DB_PATH)
results = []
run_ts  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def log(check, table, status, rows_affected, note=""):
    results.append({
        "run_at":        run_ts,
        "check_name":    check,
        "table":         table,
        "status":        status,       # PASS / FAIL / WARN
        "rows_affected": rows_affected,
        "note":          note,
    })
    icon = "✅" if status == "PASS" else ("⚠️ " if status == "WARN" else "❌")
    print(f"  {icon}  {status:<4}  {check}  ({rows_affected} rows)  {note}")

print("\n" + "="*60)
print("  CDR DATA QUALITY CHECKS")
print(f"  Run at: {run_ts}")
print("="*60 + "\n")


# ── 1. NULL BALANCE CHECK ─────────────────────────────────────────────────
print("[ accounts ]")
df = pd.read_sql("SELECT COUNT(*) AS n FROM raw_accounts WHERE balance_aud IS NULL", conn)
n = df["n"][0]
log("null_balance", "raw_accounts",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} accounts had null balance — replaced with 0.00 in clean layer")


# ── 2. FUTURE OPEN DATE CHECK ─────────────────────────────────────────────
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM raw_accounts
    WHERE DATE(open_date) > DATE('now')
""", conn)
n = df["n"][0]
log("future_open_date", "raw_accounts",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} accounts had future open_date — excluded from clean layer")


# ── 3. ORPHANED ACCOUNTS (no matching customer) ───────────────────────────
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM raw_accounts
    WHERE customer_id NOT IN (SELECT customer_id FROM raw_customers)
""", conn)
n = df["n"][0]
log("orphaned_accounts", "raw_accounts",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} accounts with no matching customer_id")

print()


# ── 4. MISSING CONSENT CUSTOMER ID ───────────────────────────────────────
print("[ consents ]")
df = pd.read_sql("SELECT COUNT(*) AS n FROM raw_consents WHERE customer_id IS NULL", conn)
n = df["n"][0]
log("null_customer_id", "raw_consents",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} consents with no customer_id — compliance risk, excluded from clean layer")


# ── 5. UNKNOWN CONSENT STATUS ────────────────────────────────────────────
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM raw_consents
    WHERE status NOT IN ('active', 'revoked', 'expired')
""", conn)
n = df["n"][0]
log("invalid_consent_status", "raw_consents",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} consents with unrecognised status value")

print()


# ── 6. DUPLICATE TRANSACTION IDS ─────────────────────────────────────────
print("[ transactions ]")
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM (
        SELECT transaction_id FROM raw_transactions
        GROUP BY transaction_id HAVING COUNT(*) > 1
    )
""", conn)
n = df["n"][0]
log("duplicate_transaction_ids", "raw_transactions",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} duplicate transaction_ids — deduplicated in clean layer")


# ── 7. NEGATIVE OR ZERO AMOUNTS ───────────────────────────────────────────
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM raw_transactions
    WHERE CAST(amount_aud AS REAL) <= 0
""", conn)
n = df["n"][0]
log("invalid_amount", "raw_transactions",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} transactions with zero or negative amount — excluded")


# ── 8. ORPHANED TRANSACTIONS (no matching account) ────────────────────────
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM raw_transactions
    WHERE account_id NOT IN (SELECT account_id FROM clean_accounts)
""", conn)
n = df["n"][0]
log("orphaned_transactions", "raw_transactions",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} transactions with no matching account_id — excluded")

print()


# ── 9. FEED RECONCILIATION BREACH ────────────────────────────────────────
print("[ feed_logs ]")
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM clean_feed_logs
    WHERE (CAST(records_expected AS REAL) - CAST(records_received AS REAL))
          / NULLIF(CAST(records_expected AS REAL), 0) > 0.05
""", conn)
n = df["n"][0]
log("reconciliation_breach", "clean_feed_logs",
    "FAIL" if n > 0 else "PASS", n,
    f"{n} feeds where records received fell >5% below expected")


# ── 10. FEED FRESHNESS (any feed older than 24 hours with no follow-up) ───
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM (
        SELECT data_holder, MAX(run_at) AS last_run
        FROM clean_feed_logs
        GROUP BY data_holder
    )
    WHERE DATETIME(last_run) < DATETIME('now', '-1 day')
""", conn)
n = df["n"][0]
log("stale_feed", "clean_feed_logs",
    "WARN" if n > 0 else "PASS", n,
    f"{n} data holders have not sent a feed in the last 24 hours")


# ── 11. NULL LATENCY ON NON-FAILED FEEDS ─────────────────────────────────
df = pd.read_sql("""
    SELECT COUNT(*) AS n FROM raw_feed_logs
    WHERE latency_ms IS NULL AND status != 'failed'
""", conn)
n = df["n"][0]
log("missing_latency", "raw_feed_logs",
    "WARN" if n > 0 else "PASS", n,
    f"{n} non-failed feeds missing latency_ms — set to -1 sentinel in clean layer")


# ── 12. ROW COUNT RECONCILIATION BETWEEN LAYERS ──────────────────────────
print()
print("[ layer row counts ]")
pairs = [
    ("raw_accounts",      "clean_accounts"),
    ("raw_transactions",  "clean_transactions"),
    ("raw_consents",      "clean_consents"),
    ("raw_feed_logs",     "clean_feed_logs"),
]
for raw, clean in pairs:
    raw_n   = pd.read_sql(f"SELECT COUNT(*) AS n FROM {raw}",   conn)["n"][0]
    clean_n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {clean}", conn)["n"][0]
    dropped = raw_n - clean_n
    pct     = round(dropped / raw_n * 100, 1) if raw_n > 0 else 0
    status  = "PASS" if pct < 10 else "WARN"
    log(f"row_count_{clean}", f"{raw}→{clean}",
        status, dropped,
        f"{raw_n} raw → {clean_n} clean ({dropped} dropped, {pct}%)")


# ── SUMMARY ───────────────────────────────────────────────────────────────
conn.close()

df_log = pd.DataFrame(results)
df_log.to_csv(LOG_PATH, index=False)

total  = len(results)
passed = len(df_log[df_log["status"] == "PASS"])
failed = len(df_log[df_log["status"] == "FAIL"])
warned = len(df_log[df_log["status"] == "WARN"])

print()
print("="*60)
print(f"  SUMMARY:  {passed} passed  |  {failed} failed  |  {warned} warnings")
print(f"  Audit log saved → {LOG_PATH}")
print("="*60 + "\n")


  CDR DATA QUALITY CHECKS
  Run at: 2026-04-20 18:28:07

[ accounts ]
  ❌  FAIL  null_balance  (9 rows)  9 accounts had null balance — replaced with 0.00 in clean layer
  ❌  FAIL  future_open_date  (2 rows)  2 accounts had future open_date — excluded from clean layer
  ✅  PASS  orphaned_accounts  (0 rows)  0 accounts with no matching customer_id

[ consents ]
  ❌  FAIL  null_customer_id  (9 rows)  9 consents with no customer_id — compliance risk, excluded from clean layer
  ✅  PASS  invalid_consent_status  (0 rows)  0 consents with unrecognised status value

[ transactions ]
  ❌  FAIL  duplicate_transaction_ids  (154 rows)  154 duplicate transaction_ids — deduplicated in clean layer
  ✅  PASS  invalid_amount  (0 rows)  0 transactions with zero or negative amount — excluded
  ❌  FAIL  orphaned_transactions  (31 rows)  31 transactions with no matching account_id — excluded

[ feed_logs ]
  ❌  FAIL  reconciliation_breach  (161 rows)  161 feeds where records received fell >5% below expect

OSError: Cannot save file into a non-existent directory: '../data'